# Small Minds · Overnight A100 AutoResearch Lab

Run **Karpathy-style autonomous architecture & optimizer discovery** on your Google AI Ultra **A100 GPU with 24-hour Background Execution** — zero Google Drive storage needed.

Every candidate genome is guaranteed **100% WebGPU-compatible** (`headDim <= 32`, even, exact `SmallMindsTransformer` kernels), trained under an identical fixed wall-clock/step budget, and graded on held-out validation **bits per byte (`val_bpb`)**.

### What it tests automatically
- **Optimizers:** Decoupled `AdamW` vs. real **Newton-Schulz `Muon`** (orthogonalized 2D matrix momentum + AdamW for 1D/embeddings)
- **Activations:** `SwiGLU` vs. `GeGLU`
- **Geometry:** Width (`dModel` 64–192), depth (`nLayers` 2–6), FFN ratio (`2×` / `2.5×` / `3×` / `4×`)
- **Positional encoding & schedules:** RoPE `θ` (`500`, `1000`, `10000`), learning rate, weight decay, cosine warmdown
- **Outputs:** Live Pareto plot + `autoresearch-results.json` + winning **`autoresearch-best.smw`** checkpoint ready to load straight into Part 3 or Part 5!

In [ ]:
#@title 1. Check GPU & install lightweight dependencies
!pip -q install -U datasets matplotlib tqdm

import os, time, math, json, copy, random, zipfile
from dataclasses import asdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
print(f"Device: {DEVICE} ({GPU_NAME}) | AMP: {AMP_DTYPE}")

In [ ]:
#@title 2. AutoResearch budget & search settings
STRATEGY = "ucb-bandit"  #@param ["ucb-bandit", "evolutionary", "hill-climb"]
NUM_TRIALS = 40  #@param {type:"integer"}
SECONDS_PER_TRIAL = 35.0  #@param {type:"number"}
SEQ_LEN = 128  #@param {type:"integer"}
BATCH_SIZE = 128  #@param {type:"integer"}
STORIES_COUNT = 40000  #@param {type:"integer"}
FINAL_POLISH_STEPS = 2500  #@param {type:"integer"}

OUT_DIR = Path("/content/small_minds_autoresearch")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 3. Bit-exact Small Minds Transformer + SMW1 Codec
"""Bit-exact PyTorch mirror of Small Minds' pure-WebGPU micro-Transformer + SMW1 codec.

Matches:
  - src/lib/llm/tokenizer.ts      (98-token printable ASCII + control CharTokenizer)
  - src/lib/llm/webgpuEngine.ts   (parameter order, RMSNorm eps=1e-5, untied embed/head)
  - src/lib/llm/shaders/ops.wgsl  (interleaved-pair RoPE, SwiGLU / tanh-approx GeGLU)
  - src/lib/llm/checkpoint.ts     (SMW1 binary header + group-wise Q8 / FP32 payload)
"""

from __future__ import annotations

import json
import math
import struct
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Literal

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

BOS_ID = 0
EOS_ID = 1
NL_ID = 2
FIRST_PRINTABLE = 3
VOCAB_SIZE = 98
MAGIC = b"SMW1"


def encode_text(text: str, add_bos: bool = False) -> np.ndarray:
    out: list[int] = [BOS_ID] if add_bos else []
    for ch in text:
        code = ord(ch)
        if code == 10:
            out.append(NL_ID)
        elif 32 <= code <= 126:
            out.append(FIRST_PRINTABLE + (code - 32))
        elif code == 9:
            out.append(FIRST_PRINTABLE)
        elif code in (8216, 8217):
            out.append(FIRST_PRINTABLE + (39 - 32))
        elif code in (8220, 8221):
            out.append(FIRST_PRINTABLE + (34 - 32))
        elif code in (8211, 8212):
            out.append(FIRST_PRINTABLE + (45 - 32))
    return np.asarray(out, dtype=np.int64)


def decode_tokens(ids: list[int] | np.ndarray | torch.Tensor) -> str:
    if isinstance(ids, torch.Tensor):
        ids = ids.detach().cpu().tolist()
    chars: list[str] = []
    for i in ids:
        idx = int(i)
        if idx in (BOS_ID, EOS_ID):
            continue
        if idx == NL_ID:
            chars.append("\n")
        elif FIRST_PRINTABLE <= idx < VOCAB_SIZE:
            chars.append(chr(32 + (idx - FIRST_PRINTABLE)))
    return "".join(chars)


@dataclass
class SmallMindsConfig:
    vocabSize: int = VOCAB_SIZE
    seqLen: int = 128
    dModel: int = 128
    nLayers: int = 4
    nHeads: int = 4
    ffnDim: int = 320
    activation: Literal["swiglu", "gelu"] = "swiglu"
    optimizer: Literal["adamw", "muon"] = "adamw"
    lr: float = 0.0008
    weightDecay: float = 0.01
    ropeTheta: float = 10000.0
    batchSize: int = 4

    def validate(self) -> None:
        assert self.vocabSize == VOCAB_SIZE, f"vocabSize must be {VOCAB_SIZE}"
        assert self.dModel % self.nHeads == 0, "dModel must be divisible by nHeads"
        hd = self.dModel // self.nHeads
        assert hd <= 32 and hd % 2 == 0, (
            f"WebGPU FlashAttention requires even headDim <= 32 (got {hd})"
        )

    def param_count(self) -> int:
        v, d, l, f = self.vocabSize, self.dModel, self.nLayers, self.ffnDim
        per_layer = d + 4 * d * d + d + 2 * f * d + d * f
        return v * d + l * per_layer + d + v * d


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xf = x.float()
        inv = torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (xf * inv).to(x.dtype) * self.weight


def precompute_rope(seq_len: int, head_dim: int, theta: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    pairs = torch.arange(0, head_dim // 2, device=device, dtype=torch.float32)
    freqs = theta ** (-2.0 * pairs / head_dim)
    positions = torch.arange(seq_len, device=device, dtype=torch.float32)
    angles = positions[:, None] * freqs[None, :]
    return torch.cos(angles)[None, :, None, :], torch.sin(angles)[None, :, None, :]


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # Interleaved adjacent pairs (2p, 2p+1) — exact match with ops.wgsl:rope_apply
    xf = x.float()
    x0 = xf[..., 0::2]
    x1 = xf[..., 1::2]
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack((out0, out1), dim=-1).flatten(-2).to(x.dtype)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: SmallMindsConfig):
        super().__init__()
        d, f = cfg.dModel, cfg.ffnDim
        self.n_heads = cfg.nHeads
        self.head_dim = d // cfg.nHeads
        self.activation = cfg.activation

        self.rms1 = RMSNorm(d)
        self.wq = nn.Linear(d, d, bias=False)
        self.wk = nn.Linear(d, d, bias=False)
        self.wv = nn.Linear(d, d, bias=False)
        self.wo = nn.Linear(d, d, bias=False)

        self.rms2 = RMSNorm(d)
        self.w1 = nn.Linear(d, f, bias=False)  # gate
        self.w3 = nn.Linear(d, f, bias=False)  # up
        self.w2 = nn.Linear(f, d, bias=False)  # down

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        h = self.rms1(x)
        q = self.wq(h).view(b, t, self.n_heads, self.head_dim)
        k = self.wk(h).view(b, t, self.n_heads, self.head_dim)
        v = self.wv(h).view(b, t, self.n_heads, self.head_dim)

        q = apply_rope(q, cos[:, :t], sin[:, :t]).transpose(1, 2)
        k = apply_rope(k, cos[:, :t], sin[:, :t]).transpose(1, 2)
        v = v.transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(b, t, d)
        x = x + self.wo(y)

        h2 = self.rms2(x)
        gate = self.w1(h2)
        up = self.w3(h2)
        if self.activation == "swiglu":
            act = F.silu(gate) * up
        else:
            act = F.gelu(gate, approximate="tanh") * up
        return x + self.w2(act)


class SmallMindsTransformer(nn.Module):
    def __init__(self, cfg: SmallMindsConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.w_embed = nn.Embedding(cfg.vocabSize, cfg.dModel)
        self.layers = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.nLayers)])
        self.rms_final = RMSNorm(cfg.dModel)
        self.w_head = nn.Linear(cfg.dModel, cfg.vocabSize, bias=False)
        self._init_weights()

        cos, sin = precompute_rope(cfg.seqLen, cfg.dModel // cfg.nHeads, cfg.ropeTheta, torch.device("cpu"))
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _init_weights(self) -> None:
        std = 1.0 / math.sqrt(self.cfg.dModel)
        res_std = std / math.sqrt(2.0 * self.cfg.nLayers)
        nn.init.normal_(self.w_embed.weight, std=0.04)
        for layer in self.layers:
            nn.init.ones_(layer.rms1.weight)
            nn.init.normal_(layer.wq.weight, std=std)
            nn.init.normal_(layer.wk.weight, std=std)
            nn.init.normal_(layer.wv.weight, std=std)
            nn.init.normal_(layer.wo.weight, std=res_std)
            nn.init.ones_(layer.rms2.weight)
            nn.init.normal_(layer.w1.weight, std=std)
            nn.init.normal_(layer.w3.weight, std=std)
            nn.init.normal_(layer.w2.weight, std=res_std)
        nn.init.ones_(self.rms_final.weight)
        nn.init.normal_(self.w_head.weight, std=std * 0.5)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.w_embed(input_ids)
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin)
        return self.w_head(self.rms_final(x))

    def ordered_params(self) -> list[nn.Parameter]:
        params: list[nn.Parameter] = [self.w_embed.weight]
        for layer in self.layers:
            params.extend([
                layer.rms1.weight,
                layer.wq.weight,
                layer.wk.weight,
                layer.wv.weight,
                layer.wo.weight,
                layer.rms2.weight,
                layer.w1.weight,
                layer.w3.weight,
                layer.w2.weight,
            ])
        params.extend([self.rms_final.weight, self.w_head.weight])
        return params

    def export_flat_weights(self) -> np.ndarray:
        chunks = [p.detach().float().cpu().contiguous().view(-1).numpy() for p in self.ordered_params()]
        flat = np.concatenate(chunks).astype("<f4", copy=False)
        return np.clip(flat, -7.99, 7.99)

    @torch.no_grad()
    def load_flat_weights(self, flat: np.ndarray) -> None:
        arr = np.asarray(flat, dtype=np.float32)
        offset = 0
        for p in self.ordered_params():
            n = p.numel()
            p.copy_(torch.from_numpy(arr[offset : offset + n]).view_as(p).to(p.device, p.dtype))
            offset += n
        assert offset == arr.size, f"Weight size mismatch: consumed {offset}, got {arr.size}"

    @torch.no_grad()
    def generate(
        self,
        prompt: str,
        max_new_tokens: int = 110,
        temperature: float = 0.68,
        top_p: float = 0.9,
        stop_on_newline: bool = True,
    ) -> str:
        self.eval()
        device = self.w_embed.weight.device
        ctx = encode_text(prompt).tolist() or [BOS_ID]
        gen: list[int] = []
        for _ in range(max_new_tokens):
            window = ctx[-self.cfg.seqLen :]
            inp = torch.tensor([window], dtype=torch.long, device=device)
            logits = self(inp)[0, -1].float() / max(0.05, temperature)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cdf = torch.cumsum(sorted_probs, dim=-1)
            mask = cdf - sorted_probs > top_p
            sorted_probs[mask] = 0.0
            sorted_probs /= sorted_probs.sum().clamp_min(1e-12)
            pick = int(sorted_idx[torch.multinomial(sorted_probs, 1).item()].item())
            if stop_on_newline and pick in (NL_ID, EOS_ID):
                break
            ctx.append(pick)
            gen.append(pick)
        return decode_tokens(gen)


def save_smw(
    path: str | Path,
    model: SmallMindsTransformer,
    meta: dict[str, Any] | None = None,
    quant: Literal["q8", "fp32"] = "q8",
    group_size: int = 32,
) -> dict[str, Any]:
    flat = model.export_flat_weights()
    param_count = int(flat.size)
    assert param_count == model.cfg.param_count()

    header = {
        "version": 1,
        "quant": quant,
        "groupSize": group_size,
        "paramCount": param_count,
        "config": asdict(model.cfg),
        "meta": meta or {},
    }
    header_bytes = json.dumps(header, separators=(",", ":")).encode("utf-8")
    data_offset = (8 + len(header_bytes) + 3) & ~3
    pad = b"\x00" * (data_offset - 8 - len(header_bytes))

    out_path = Path(path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if quant == "fp32":
        payload = flat.astype("<f4").tobytes()
        rmse = 0.0
    else:
        num_groups = math.ceil(param_count / group_size)
        padded_len = num_groups * group_size
        padded = np.zeros(padded_len, dtype=np.float32)
        padded[:param_count] = flat
        groups = padded.reshape(num_groups, group_size)
        scales = np.maximum(np.max(np.abs(groups), axis=1) / 127.0, 1e-8).astype("<f4")
        q = np.clip(np.rint(groups / scales[:, None]), -127, 127).astype(np.int8)
        deq = (q.astype(np.float32) * scales[:, None]).reshape(-1)[:param_count]
        rmse = float(np.sqrt(np.mean((flat - deq) ** 2)))
        payload = scales.tobytes() + q.reshape(-1)[:param_count].tobytes()

    blob = MAGIC + struct.pack("<I", len(header_bytes)) + header_bytes + pad + payload
    out_path.write_bytes(blob)
    return {"path": str(out_path), "bytes": len(blob), "param_count": param_count, "rmse": rmse}


def load_smw(path: str | Path) -> tuple[SmallMindsConfig, np.ndarray, dict[str, Any]]:
    raw = Path(path).read_bytes()
    assert raw[:4] == MAGIC, "Bad SMW1 magic"
    header_len = struct.unpack_from("<I", raw, 4)[0]
    header = json.loads(raw[8 : 8 + header_len].decode("utf-8"))
    cfg = SmallMindsConfig(**header["config"])
    param_count = int(header["paramCount"])
    data_offset = (8 + header_len + 3) & ~3

    if header["quant"] == "fp32":
        weights = np.frombuffer(raw, dtype="<f4", count=param_count, offset=data_offset).copy()
    else:
        gs = int(header.get("groupSize", 32))
        num_groups = math.ceil(param_count / gs)
        scales = np.frombuffer(raw, dtype="<f4", count=num_groups, offset=data_offset)
        q_offset = data_offset + num_groups * 4
        q = np.frombuffer(raw, dtype=np.int8, count=param_count, offset=q_offset)
        weights = np.empty(param_count, dtype=np.float32)
        for g in range(num_groups):
            s = g * gs
            e = min(param_count, s + gs)
            weights[s:e] = q[s:e].astype(np.float32) * scales[g]
    return cfg, weights, header.get("meta", {})

In [ ]:
#@title 4. Real Newton-Schulz Muon Optimizer (for 2D projections) + Streamed Dataset
from datasets import load_dataset
from tqdm.auto import tqdm

@torch.no_grad()
def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    '''Orthogonalize 2D gradient updates using Keller Jordan's Muon Newton-Schulz iteration.'''
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.bfloat16()
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if transposed:
        X = X.T
    return X.to(G.dtype)

class MuonHybridOptimizer:
    '''Uses Newton-Schulz Muon on 2D Transformer body weights + AdamW on embeddings/head/norms.'''
    def __init__(self, model: SmallMindsTransformer, lr: float, weight_decay: float):
        muon_params, adam_decay, adam_nodecay = [], [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if p.ndim == 2 and ("w_embed" not in name) and ("w_head" not in name):
                muon_params.append(p)
            elif p.ndim >= 2:
                adam_decay.append(p)
            else:
                adam_nodecay.append(p)

        self.muon_params = muon_params
        self.muon_bufs = [torch.zeros_like(p) for p in muon_params]
        self.base_lr = lr
        self.muon_lr = lr * 8.0  # Muon naturally prefers higher LR than AdamW
        self.wd = weight_decay
        self.adam = torch.optim.AdamW(
            [{"params": adam_decay, "weight_decay": weight_decay}, {"params": adam_nodecay, "weight_decay": 0.0}],
            lr=lr, betas=(0.9, 0.95), fused=(DEVICE.type == "cuda"),
        )

    def set_lr_scale(self, scale: float) -> None:
        self.muon_lr = self.base_lr * 8.0 * scale
        for pg in self.adam.param_groups:
            pg["lr"] = self.base_lr * scale

    def zero_grad(self) -> None:
        self.adam.zero_grad(set_to_none=True)
        for p in self.muon_params:
            p.grad = None

    @torch.no_grad()
    def step(self) -> None:
        self.adam.step()
        beta = 0.95
        for p, buf in zip(self.muon_params, self.muon_bufs):
            if p.grad is None:
                continue
            g = p.grad
            buf.lerp_(g, 1.0 - beta)
            nesterov = g.lerp(buf, beta)
            orth = zeropower_via_newtonschulz5(nesterov, steps=5)
            rms_scale = max(1.0, p.size(0) / p.size(1)) ** 0.5
            p.mul_(1.0 - self.muon_lr * self.wd * 0.25)
            p.add_(orth, alpha=-self.muon_lr * rms_scale)

print(f"Streaming {STORIES_COUNT:,} stories for fixed-split AutoResearch evaluation...")
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
texts = []
for i, row in enumerate(tqdm(ds, total=STORIES_COUNT, desc="Loading corpus")):
    if i >= STORIES_COUNT:
        break
    texts.append(row["text"].strip().lower() + "\n\n")

tokens = encode_text("".join(texts))
del texts
split = int(len(tokens) * 0.95)
train_toks = torch.from_numpy(tokens[:split]).to(torch.long)
val_toks = torch.from_numpy(tokens[split:]).to(torch.long)
print(f"Dataset ready: {len(train_toks):,} train | {len(val_toks):,} val tokens")

def sample_batch(data: torch.Tensor, batch_size: int, seq_len: int, gen: torch.Generator | None = None):
    ix = torch.randint(0, len(data) - seq_len - 1, (batch_size,), generator=gen)
    x = torch.stack([data[i : i + seq_len] for i in ix]).to(DEVICE, non_blocking=True)
    y = torch.stack([data[i + 1 : i + seq_len + 1] for i in ix]).to(DEVICE, non_blocking=True)
    return x, y

@torch.no_grad()
def fixed_val_bpb(model: nn.Module, seq_len: int, batches: int = 24) -> tuple[float, float]:
    model.eval()
    g = torch.Generator().manual_seed(1337)
    losses = []
    for _ in range(batches):
        x, y = sample_batch(val_toks, 64, seq_len, gen=g)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        losses.append(loss.item())
    model.train()
    mean_loss = float(np.mean(losses))
    return mean_loss, mean_loss / math.log(2.0)

In [ ]:
#@title 5. Autonomous Research Loop (Fixed Time Budget per Trial)

# Every (dModel, nHeads) pair below guarantees even headDim <= 32 for WebGPU FlashAttention!
WIDTH_HEADS = [(64, 4), (96, 4), (128, 4), (160, 8), (192, 6)]

BASELINE_GENOME = {
    "dModel": 128,
    "nHeads": 4,
    "nLayers": 4,
    "ffnMult": 2.5,
    "activation": "gelu",
    "optimizer": "adamw",
    "lr": 0.002,
    "weightDecay": 0.02,
    "ropeTheta": 10000.0,
    "cosineWarmdown": False,
}

def genome_to_cfg(g: dict) -> SmallMindsConfig:
    # Snap ffnDim to multiple of 16 for optimal WebGPU 16x16 tiled SGEMM
    ffn_dim = max(64, int(round((g["dModel"] * g["ffnMult"]) / 16)) * 16)
    return SmallMindsConfig(
        vocabSize=VOCAB_SIZE,
        seqLen=SEQ_LEN,
        dModel=g["dModel"],
        nLayers=g["nLayers"],
        nHeads=g["nHeads"],
        ffnDim=ffn_dim,
        activation=g["activation"],
        optimizer=g["optimizer"],
        lr=g["lr"],
        weightDecay=g["weightDecay"],
        ropeTheta=g["ropeTheta"],
        batchSize=4,
    )

def propose_mutations(g: dict) -> list[tuple[str, str, dict]]:
    out = []
    # 1. Activation
    nxt = "swiglu" if g["activation"] == "gelu" else "gelu"
    out.append(("activation", f"activation: {g['activation']} -> {nxt}", {**g, "activation": nxt}))
    # 2. Optimizer (AdamW vs real Newton-Schulz Muon!)
    nxt_o = "muon" if g["optimizer"] == "adamw" else "adamw"
    out.append(("optimizer", f"optimizer: {g['optimizer']} -> {nxt_o}", {**g, "optimizer": nxt_o}))
    # 3. Cosine warmdown
    nxt_c = not g["cosineWarmdown"]
    out.append(("schedule", f"cosineWarmdown: {g['cosineWarmdown']} -> {nxt_c}", {**g, "cosineWarmdown": nxt_c}))
    # 4. Learning rate up / down
    up_lr = round(min(0.008, g["lr"] * 1.45), 5)
    if up_lr != g["lr"]:
        out.append(("lr_up", f"lr: {g['lr']} -> {up_lr}", {**g, "lr": up_lr}))
    dn_lr = round(max(0.0004, g["lr"] * 0.68), 5)
    if dn_lr != g["lr"]:
        out.append(("lr_down", f"lr: {g['lr']} -> {dn_lr}", {**g, "lr": dn_lr}))
    # 5. Width
    cur_idx = [w for w, _ in WIDTH_HEADS].index(g["dModel"])
    for delta in (-1, 1):
        ni = cur_idx + delta
        if 0 <= ni < len(WIDTH_HEADS):
            nw, nh = WIDTH_HEADS[ni]
            out.append(("width", f"dModel: {g['dModel']} -> {nw} ({nh}H)", {**g, "dModel": nw, "nHeads": nh}))
    # 6. Depth
    for delta in (-1, 1):
        nl = g["nLayers"] + delta
        if 2 <= nl <= 6:
            out.append(("depth", f"nLayers: {g['nLayers']} -> {nl}", {**g, "nLayers": nl}))
    # 7. FFN multiplier
    for m in (2.0, 2.5, 3.0, 4.0):
        if m != g["ffnMult"]:
            out.append(("ffn_ratio", f"ffnMult: {g['ffnMult']}x -> {m}x", {**g, "ffnMult": m}))
    # 8. RoPE theta
    for th in (500.0, 1000.0, 10000.0):
        if th != g["ropeTheta"]:
            out.append(("rope_theta", f"ropeTheta: {int(g['ropeTheta'])} -> {int(th)}", {**g, "ropeTheta": th}))
    # 9. Weight decay
    for wd in (0.005, 0.02, 0.08):
        if wd != g["weightDecay"]:
            out.append(("weight_decay", f"weightDecay: {g['weightDecay']} -> {wd}", {**g, "weightDecay": wd}))
    return out

def run_trial(genome: dict, seconds_budget: float, seed: int = 2026) -> dict:
    cfg = genome_to_cfg(genome)
    torch.manual_seed(seed)
    model = SmallMindsTransformer(cfg).to(DEVICE)

    if genome["optimizer"] == "muon":
        opt = MuonHybridOptimizer(model, lr=genome["lr"], weight_decay=genome["weightDecay"])
    else:
        decay = [p for n, p in model.named_parameters() if p.dim() >= 2]
        nodecay = [p for n, p in model.named_parameters() if p.dim() < 2]
        opt = torch.optim.AdamW(
            [{"params": decay, "weight_decay": genome["weightDecay"]}, {"params": nodecay, "weight_decay": 0.0}],
            lr=genome["lr"], betas=(0.9, 0.95), fused=(DEVICE.type == "cuda"),
        )

    batch_gen = torch.Generator().manual_seed(918273)
    t_start = time.time()
    steps = 0
    last_loss = 4.6

    while True:
        elapsed = time.time() - t_start
        if steps > 0 and elapsed >= seconds_budget:
            break
        frac = min(1.0, elapsed / max(1.0, seconds_budget))
        if frac < 0.06:
            scale = max(0.1, frac / 0.06)
        elif genome["cosineWarmdown"] and frac > 0.65:
            wd_f = (frac - 0.65) / 0.35
            scale = 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * wd_f))
        else:
            scale = 1.0

        if isinstance(opt, MuonHybridOptimizer):
            opt.set_lr_scale(scale)
        else:
            for pg in opt.param_groups:
                pg["lr"] = genome["lr"] * scale

        x, y = sample_batch(train_toks, BATCH_SIZE, SEQ_LEN, gen=batch_gen)
        opt.zero_grad() if isinstance(opt, MuonHybridOptimizer) else opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        last_loss = float(loss.item())
        steps += 1

    wall_s = time.time() - t_start
    val_loss, val_bpb = fixed_val_bpb(model, SEQ_LEN)
    sample = model.generate("once upon a time, ", max_new_tokens=64, temperature=0.7, stop_on_newline=False)
    tok_per_sec = (steps * BATCH_SIZE * SEQ_LEN) / max(wall_s, 1e-3)
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    del model, opt
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "cfg": asdict(cfg),
        "paramCount": cfg.param_count(),
        "steps": steps,
        "wallSec": round(wall_s, 2),
        "tokPerSec": round(tok_per_sec, 1),
        "trainLoss": round(last_loss, 4),
        "valLoss": round(val_loss, 4),
        "valBpb": round(val_bpb, 4),
        "sample": sample,
        "state": state,
    }

history = []
axis_stats: dict[str, list[float]] = {}
best_node = None
best_state = None
seen_sigs = set()

for trial in range(1, NUM_TRIALS + 1):
    if trial == 1:
        parent_id = None
        axis = "baseline"
        diff = "init baseline (d128 L4 gelu adamw)"
        genome = copy.deepcopy(BASELINE_GENOME)
    else:
        if STRATEGY == "evolutionary" and len(history) >= 3:
            kept = sorted([h for h in history if h["status"] != "discarded"], key=lambda n: n["valBpb"])
            parent = random.choice(kept[: min(3, len(kept))])
        else:
            parent = best_node
        parent_id = parent["id"]
        cands = propose_mutations(parent["genome"])
        unseen = [c for c in cands if json.dumps(c[2], sort_keys=True) not in seen_sigs]
        pool = unseen if unseen else cands

        if STRATEGY == "ucb-bandit":
            total_t = max(1, len(history))
            scored = []
            for a, d, g_cand in pool:
                rewards = axis_stats.get(a, [])
                mean_r = (sum(rewards) / len(rewards)) if rewards else 0.25
                bonus = math.sqrt((1.4 * math.log(total_t + 1)) / (len(rewards) + 0.5))
                scored.append((mean_r + bonus + random.random() * 0.04, a, d, g_cand))
            scored.sort(key=lambda t: t[0], reverse=True)
            _, axis, diff, genome = scored[0]
        else:
            axis, diff, genome = random.choice(pool)

    seen_sigs.add(json.dumps(genome, sort_keys=True))
    res = run_trial(genome, SECONDS_PER_TRIAL, seed=2026 + trial * 17)

    prev_best = best_node["valBpb"] if best_node else res["valBpb"]
    delta = round(res["valBpb"] - prev_best, 4)
    improved = (trial == 1) or (res["valBpb"] < prev_best - 0.002)
    status = "baseline" if trial == 1 else ("kept" if improved else "discarded")

    if trial > 1:
        axis_stats.setdefault(axis, []).append(max(0.0, -delta * 4.0))

    node = {
        "id": trial,
        "parentId": parent_id,
        "status": status,
        "mutationAxis": axis,
        "diff": diff,
        "genome": genome,
        **{k: v for k, v in res.items() if k != "state"},
        "deltaBpb": delta,
    }
    history.append(node)
    if improved:
        best_node = node
        best_state = res["state"]

    # Persist incrementally every trial (safe for background execution!)
    (OUT_DIR / "autoresearch-results.json").write_text(
        json.dumps({"gpu": GPU_NAME, "strategy": STRATEGY, "best": best_node, "history": history}, indent=2),
        encoding="utf-8",
    )
    badge = "BASE" if status == "baseline" else ("KEEP" if status == "kept" else "DROP")
    print(f"[{badge}] #{trial:02d} {diff:<36} | {node['paramCount']/1e3:6.1f}K | {node['steps']:4d} steps | val {node['valBpb']:.4f} bpb ({delta:+.4f})")

print(f"\nWinner: #{best_node['id']} {best_node['diff']} -> val_bpb={best_node['valBpb']:.4f} ({best_node['paramCount']:,} params)")

In [ ]:
#@title 6. Plot Pareto Frontier & Polish Winning Model -> Export `.smw` + Download `.zip`
xs = [h["id"] for h in history]
ys = [h["valBpb"] for h in history]
best_curve = []
cur = 99.0
for y in ys:
    cur = min(cur, y)
    best_curve.append(cur)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), facecolor="#0a0c14")
for ax in (ax1, ax2):
    ax.set_facecolor("#111524")
    ax.tick_params(colors="#b8c3e0")
    for sp in ax.spines.values():
        sp.set_color("#2b3450")

colors = ["#5ef0b5" if h["status"] != "discarded" else "#677294" for h in history]
ax1.scatter(xs, ys, c=colors, s=55, zorder=3)
ax1.plot(xs, best_curve, color="#5ef0b5", linewidth=2.2, label="Frontier best val_bpb")
ax1.set_title(f"AutoResearch Frontier on {GPU_NAME}", color="#eef3ff")
ax1.set_xlabel("Trial #", color="#b8c3e0")
ax1.set_ylabel("Validation bits/byte (lower is better)", color="#b8c3e0")
ax1.legend(facecolor="#111524", edgecolor="#2b3450", labelcolor="#eef3ff")

axes_names = sorted(axis_stats.keys(), key=lambda k: sum(axis_stats[k]) / max(1, len(axis_stats[k])), reverse=True)
mean_rewards = [sum(axis_stats[k]) / max(1, len(axis_stats[k])) for k in axes_names]
ax2.barh(axes_names, mean_rewards, color="#7aa2ff")
ax2.invert_yaxis()
ax2.set_title("Mean Improvement Reward by Mutation Axis", color="#eef3ff")
ax2.set_xlabel("Reward", color="#b8c3e0")
plt.tight_layout()
plt.savefig(OUT_DIR / "autoresearch-frontier.png", dpi=150, facecolor="#0a0c14")
plt.show()

# Polish the winning architecture for extra steps so autoresearch-best.smw is ready to chat/sample in WebGPU!
win_cfg = genome_to_cfg(best_node["genome"])
win_model = SmallMindsTransformer(win_cfg).to(DEVICE)
win_model.load_state_dict(best_state)
if FINAL_POLISH_STEPS > 0:
    print(f"Polishing winning architecture (#{best_node['id']}) for {FINAL_POLISH_STEPS} more steps...")
    g = best_node["genome"]
    opt = MuonHybridOptimizer(win_model, lr=g["lr"] * 0.5, weight_decay=g["weightDecay"]) if g["optimizer"] == "muon" else torch.optim.AdamW(win_model.parameters(), lr=g["lr"] * 0.5, weight_decay=g["weightDecay"], fused=(DEVICE.type == "cuda"))
    for step in range(1, FINAL_POLISH_STEPS + 1):
        scale = 0.08 + 0.92 * 0.5 * (1.0 + math.cos(math.pi * step / FINAL_POLISH_STEPS))
        if isinstance(opt, MuonHybridOptimizer):
            opt.set_lr_scale(scale)
        else:
            for pg in opt.param_groups:
                pg["lr"] = g["lr"] * 0.5 * scale
        x, y = sample_batch(train_toks, BATCH_SIZE, SEQ_LEN)
        opt.zero_grad() if isinstance(opt, MuonHybridOptimizer) else opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
            loss = F.cross_entropy(win_model(x).view(-1, VOCAB_SIZE), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(win_model.parameters(), 1.0)
        opt.step()

final_loss, final_bpb = fixed_val_bpb(win_model, SEQ_LEN)
meta = {
    "speciesId": "autoresearch",
    "label": f"AutoResearch Winner (#{best_node['id']})",
    "trainedOn": f"Colab {GPU_NAME} AutoResearch",
    "trainedSteps": best_node["steps"] + FINAL_POLISH_STEPS,
    "valLoss": round(final_loss, 4),
    "valBpb": round(final_bpb, 4),
    "corpus": "TinyStories AutoResearch",
    "createdAt": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
smw_info = save_smw(OUT_DIR / "autoresearch-best.smw", win_model, meta=meta, quant="q8")
print(f"Exported {smw_info['path']} ({smw_info['bytes']/1024:.1f} KB Q8 | final val {final_bpb:.4f} bpb)")
print("Sample ->", win_model.generate("once upon a time, a tiny star ", max_new_tokens=110, stop_on_newline=False))

zip_path = Path("/content/small-minds-autoresearch.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in ["autoresearch-best.smw", "autoresearch-results.json", "autoresearch-frontier.png"]:
        p = OUT_DIR / fname
        if p.exists():
            zf.write(p, arcname=fname)

print(f"\nReady to download: {zip_path} ({zip_path.stat().st_size/1024:.1f} KB)")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print(f"(Not in Colab UI; download manually from {zip_path})")